# K_01b – Zonenmodell Erweitert (6 Zonen)

**Grid-Arbitrage** · Erweitertes Netz-Zonenmodell Schweiz — 6 statt 5 Zonen

**Gruppe:** SC26_Gruppe_2 | **Status:** Kür – explorative Erweiterung | **Datum:** April 2026

---

> 🔬 **Zweck dieses Notebooks:** Granularere Zonenstruktur für präzisere BVI-Analyse.
> Ersetzt K_01 *nicht* — läuft parallel, teilt dieselben Rohdaten.
> Keine Referenz in K_00 Business Strategy. Config-Integration: nach Projekt-Abgabe.

---
*Voraussetzung: K_01 muss zuvor ausgeführt worden sein (BFE-Daten + BFS-Kantondaten im Cache).*


| [← K_01 Räumliche Analyse](K_01_Raeumliche_Analyse.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_01b'></a>

1. [Initialisierung](#initialisierung_K_01b)
2. [⚠ Zonenmodell-Warnung & lokale Definitionen](#warnung_K_01b)
3. [Daten laden (aus K_01-Cache)](#3-daten-laden_K_01b)
4. [Neue Zonenzuweisung (6 Zonen)](#4-zonenzuweisung_K_01b)
5. [Zonenbilanzen & BVI](#5-bilanzen-bvi_K_01b)
6. [Vergleich: 5-Zonen (K_01) vs. 6-Zonen (K_01b)](#6-vergleich_K_01b)
7. [⚙ Config-Dokumentation: Was in config.json gehört](#7-config-doku_K_01b)
8. [Abschluss](#abschluss_K_01b)


---
## Initialisierung<a id='initialisierung_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

Standard-Ladesequenz: Libraries, `../sync/config.json` (nur lesend), Farb- und Stilkonstanten.
Alle Werte aus config.json — keine lokalen Hardcodes ausser im explizit markierten Zonenblock (Sektion 2).


**Imports & Versionsanzeige:**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ⚙ PFAD-KONFIGURATION — experimental/ Notebook
# Alle Pfade relativ zum Notebook-Verzeichnis (experimental/)
# NICHT anpassen: PROD_DATA_DIR (nur lesen, keine Writes)
# ══════════════════════════════════════════════════════════════════════════════
import os

# Lokale experimental-Verzeichnisse (Schreiben)
EXP_BASE       = '.'                                    # ⚙ Notebook-Verzeichnis
EXP_DATA_DIR   = os.path.join(EXP_BASE, 'data', 'raw') # ⚙ Downloads + Rohdaten
EXP_INTER_DIR  = os.path.join(EXP_BASE, 'data', 'intermediate')  # ⚙ Aufbereitungen
EXP_OUTPUT_DIR = os.path.join(EXP_BASE, 'output', 'charts')      # ⚙ GIFs + Charts

# Produktions-Daten (nur lesen! keine Downloads, keine Writes)
PROD_BASE      = '../'                                # ⚙ Projektroot (2 Ebenen über experimental/)
PROD_DATA_DIR  = os.path.join(PROD_BASE, 'data', 'raw')          # ⚙ BFE, BFS, Kantone
PROD_INTER_DIR = os.path.join(PROD_BASE, 'data', 'intermediate') # ⚙ transfer.json etc.
PROD_SYNC_DIR  = os.path.join(PROD_BASE, 'sync')                 # ⚙ config.json

for d in [EXP_DATA_DIR, EXP_INTER_DIR, EXP_OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Granularität: räumliche Auflösung ────────────────────────────────────────
# ⚙ Wechseln für feinere Analyse
# 'kantone'  → 26 Kantone    (Standard, BFS PXWeb + swisstopo)
# 'bezirke'  → ~150 Bezirke  (BFS PXWeb + swissBOUNDARIES3D)
# 'gemeinden'→ ~2134 Gemeinden (BFS PXWeb + swissBOUNDARIES3D)
# Für andere Länder: 'nuts2' / 'nuts3' (Eurostat NUTS-Ebenen)
GRANULARITY = 'kantone'  # ⚙

# ── Land-Konfiguration ───────────────────────────────────────────────────────
# ⚙ Für andere Länder: CC_CODE + REGION anpassen, dann Notebook neu ausführen
CC_CODE  = 'CH'          # ⚙ ISO 3166-1 Alpha-2
# Für Europa-Erweiterung via Eurostat NUTS: CC_CODE='DE' → NUTS-3 (401 Kreise)
# Bidding-Zone für spätere ENTSO-E-Anbindung:
ENTSOE_BZ = '10YCH-SWISSGRIDZ'  # ⚙ CH; DE='10Y1001A1001A83F'; AT='10YAT-APG------L'

print(f"⚙ Pfade initialisiert:")
print(f"  EXP_DATA_DIR   = {os.path.abspath(EXP_DATA_DIR)}")
print(f"  EXP_OUTPUT_DIR = {os.path.abspath(EXP_OUTPUT_DIR)}")
print(f"  PROD_DATA_DIR  = {os.path.abspath(PROD_DATA_DIR)}")
print(f"  GRANULARITY    = {GRANULARITY}")
print(f"  CC_CODE        = {CC_CODE} | ENTSOE_BZ = {ENTSOE_BZ}")


In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import os, warnings, json
import subprocess, sys
from datetime import datetime
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors  as mcolors
import matplotlib.ticker  as mticker
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib import cm
import requests
warnings.filterwarnings('ignore')

try:
    import geopandas as gpd
    HAS_GPD = True
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','geopandas','-q'])
    import geopandas as gpd
    HAS_GPD = True

print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"GeoPandas   : {gpd.__version__}")
print(f"Matplotlib  : {plt.matplotlib.__version__}")
print(f"📅 Zuletzt ausgeführt: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")


**Konfiguration laden (SSOT — nur lesend):**

In [ ]:
# ── ../sync/config.json laden (SSOT — NUR LESEN, NICHT SCHREIBEN) ───────────
with open('../sync/config.json') as _f: CFG = json.load(_f)

MODE       = CFG['mode']
GZ_MODE    = CFG['szenarien']['gleichzeitigkeit_aktiv']
SZ_AKTIV   = GZ_MODE
# Produktionsdaten lesen (kein Download)
DATA_DIR   = PROD_DATA_DIR
INTER_DIR  = PROD_INTER_DIR
# Outputs → lokaler experimental/ Ordner
EXP_CHARTS_DIR = EXP_OUTPUT_DIR
CHARTS_DIR     = EXP_OUTPUT_DIR  # Für Abwärtskompatibilität

DPI        = CFG['visualisierung']['output_dpi']  # SSOT

# ── Farben & Stil aus config.json ────────────────────────────────────────────
_viz         = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK      = _viz.get('bg_dark',    '#0d1117')
BG_PANEL     = _viz.get('bg_panel',   '#141414')
C_PRICE      = _viz.get('c_price',    '#FFA726')
C_LOAD       = _viz.get('c_load',     '#66BB6A')
C_CHARGE     = _viz.get('c_charge',   '#1565C0')
C_FEED       = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS   = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')
C_TICK       = _viz.get('c_tick',       '#bbbbbb')
C_SPINE      = _viz.get('c_spine',      '#333333')
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')
C_SOLAR      = _viz.get('c_solar',      '#FDD835')
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')
_stil              = CFG.get('visualisierung', {}).get('stil', {})
LW                 = _stil.get('linienbreite_standard', 1.5)
ALPHA_FLAECHE      = _stil.get('alpha_flaeche',         0.12)
ALPHA_LEGENDE      = _stil.get('alpha_legende',         0.30)
FS_TITEL           = _stil.get('schriftgroesse_titel',  13)
FS_ACHSE           = _stil.get('schriftgroesse_achse',  10)
FS_TICK            = _stil.get('schriftgroesse_tick',    9)
FS_LEGENDE         = _stil.get('schriftgroesse_legende', 8)
FS_KLEIN           = _stil.get('schriftgroesse_klein',   7)

import matplotlib as mpl
mpl.rcParams.update({'figure.facecolor': BG_DARK,'axes.facecolor': BG_PANEL,
    'axes.edgecolor': C_SPINE,'axes.labelcolor': C_ACHSE,'text.color': 'white',
    'xtick.color': C_TICK,'ytick.color': C_TICK,'lines.linewidth': LW,
    'legend.facecolor': C_LEGENDE_BG,'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize': FS_LEGENDE,'legend.edgecolor': C_SPINE})
print(f"Config geladen | Mode: {MODE} | Szenario: {SZ_AKTIV}")


---
## ⚠ Zonenmodell-Warnung & lokale Definitionen<a id='warnung_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)


---

> # ⚠️ ACHTUNG: PROVISORISCHE LOKALE ZONENDEFINITION
>
> **Dieses Notebook definiert ein alternatives 6-Zonen-Modell LOKAL — ohne `../sync/config.json` zu verändern.**
>
> ### Was hier passiert:
> - Die Variablen `KANTON_TO_ZONE`, `ZONE_COLORS` und `ENGPASS_MULT` aus K_01 werden **NICHT überschrieben**
> - Stattdessen werden neue Variablen `KANTON_TO_ZONE_B`, `ZONE_COLORS_B`, `ENGPASS_MULT_B` definiert
> - Alle darauf basierenden DataFrames heissen `df_zones_b` (nie `df_zones`)
> - Alle Charts bekommen den Prefix `kuer_k01b_` (nie `kuer_k01_`)
>
> ### Warum keine Config-Änderung?
> Das 6-Zonen-Modell ist explorativ. Ob es das Standardmodell ersetzt, wird **nach der Projektabgabe** entschieden.
> Eine vorzeitige Config-Änderung würde K_01, K_00 und alle BVI-Referenzen in anderen Kür-Notebooks brechen.
>
> ### Was ⚙ bedeutet:
> Alle Werte die **später in `config.json`** gehören, sind mit `⚙` markiert.
> Sektion 7 dokumentiert vollständig welche Keys und Strukturen dafür benötigt werden.
>
> ### K_00 Business Strategy:
> **Dieses Notebook wird NICHT in K_00 referenziert.**
> Es ist eine technische Erweiterung für spätere Due-Diligence-Analysen.

---


---
## 3. Daten laden (aus K_01-Cache)<a id='3-daten-laden_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

K_01 hat die BFE-Rohdaten heruntergeladen und KANTON_POP aus BFS geladen.
Dieses Notebook liest dieselben Cache-Dateien — kein Re-Download.


In [ ]:
# ── BFE Anlagen aus Cache ────────────────────────────────────────────────────
# 📡 Liest aus Produktionsdaten (K_01 muss zuvor ausgeführt worden sein)
BFE_FILE = os.path.join(PROD_DATA_DIR, 'bfe_produktionsanlagen.gpkg')

if not os.path.exists(BFE_FILE):
    print('⚠️  BFE-Cache fehlt — bitte zuerst K_01 ausführen.')
    gdf_plants = None
else:
    try:
        layers = gpd.list_layers(BFE_FILE)
        layer  = layers['name'].iloc[0] if hasattr(layers,'__len__') else layers[0][0]
        gdf_plants = gpd.read_file(BFE_FILE, layer=layer)
        if gdf_plants.crs and gdf_plants.crs.to_epsg() != 4326:
            gdf_plants = gdf_plants.to_crs(epsg=4326)
        print(f'BFE geladen: {gdf_plants.shape[0]:,} Anlagen | CRS: {gdf_plants.crs}')
    except Exception as e:
        print(f'⚠️  BFE Ladefehler: {e}')
        gdf_plants = None

# ── Spalten-Mapping (identisch K_01) ─────────────────────────────────────────
def find_col(df, *kws):
    for kw in kws:
        for c in df.columns:
            if kw.lower() in c.lower(): return c
    return None

if gdf_plants is not None:
    COL = {
        'subcat'  : find_col(gdf_plants,'subcat','subcategor'),
        'maincat' : find_col(gdf_plants,'maincat','maincategor'),
        'leistung': find_col(gdf_plants,'totalpower','power','leistung'),
        'kanton'  : find_col(gdf_plants,'canton','kanton'),
    }
    SUBCAT_MAP = {'subcat_1':'Wasserkraft','subcat_2':'Solar','subcat_3':'Wind',
        'subcat_4':'Biomasse','subcat_5':'Geothermie','subcat_6':'Kernkraft',
        'subcat_7':'Erdoel','subcat_8':'Erdgas','subcat_9':'Kohle','subcat_10':'Abfall'}
    MAINCAT_MAP = {'maincat_1':'Wasserkraft','maincat_2':'Solar',
        'maincat_3':'Kernkraft','maincat_4':'Erdgas'}

    def _map_et(row):
        if COL['subcat'] and not pd.isna(row.get(COL['subcat'],'NaN')):
            c = str(row[COL['subcat']]).strip().lower()
            if c in SUBCAT_MAP: return SUBCAT_MAP[c]
        if COL['maincat'] and not pd.isna(row.get(COL['maincat'],'NaN')):
            c = str(row[COL['maincat']]).strip().lower()
            if c in MAINCAT_MAP: return MAINCAT_MAP[c]
        return 'Andere'

    gdf_plants['ET_group'] = gdf_plants.apply(_map_et, axis=1)
    if COL['leistung']:
        gdf_plants['kw'] = pd.to_numeric(gdf_plants[COL['leistung']], errors='coerce').fillna(0)
    print(f'ET-Mapping: {gdf_plants["ET_group"].value_counts().to_dict()}')


In [ ]:
# ── Bevölkerungsdaten nach GRANULARITY-Schalter ─────────────────────────────
# ⚙ GRANULARITY steuert räumliche Auflösung (gesetzt in Pfad-Konfiguration)
# 'kantone'   → 26 Kantone (Standard)
# 'bezirke'   → ~150 Bezirke (nächster Schritt, selbe PXWeb-API)
# 'gemeinden' → ~2134 Gemeinden (selbe API, feinste CH-Ebene)
# Für andere Länder: Eurostat demo_pjan API (NUTS-2/3)
# ── Aktuell: Kantone aus PROD-Cache (K_01 muss zuvor ausgeführt worden sein) ─

BFS_FILE   = os.path.join(PROD_INTER_DIR, 'bfs_kantone.csv')
BFS_PORTAL = 'https://www.data.bfs.admin.ch/?dataType=px&dataNumber=900010'

# Granularitäts-Hinweis
if GRANULARITY != 'kantone':
    print(f"ℹ️  GRANULARITY='{GRANULARITY}' — noch nicht implementiert in dieser Version.")
    print(f"   Fallback auf 'kantone'. Für Bezirke/Gemeinden: BFS STATPOP px-x-0102010000_101")
    print(f"   mit Filterwert auf Bezirks- oder Gemeindeebene (gleiche PXWeb-Struktur).")
    print(f"   Für Europa: Eurostat demo_pjan API, parameter geo=CH, NUTS-Ebene 2 oder 3.")
    print()

KANTON_POP_FALLBACK = {
    'ZH':1593583,'BE':1065607,'LU':433654,'UR':37208,'SZ':165539,'OW':39028,
    'NW':43921,'GL':40590,'ZG':130426,'FR':340765,'SO':279375,'BS':183709,
    'BL':297025,'SH':86928,'AR':58050,'AI':16293,'SG':530468,'GR':202461,
    'AG':718084,'TG':295373,'TI':358353,'VD':826400,'VS':357043,'NE':177589,
    'GE':511655,'JU':73584
}

if os.path.exists(BFS_FILE):
    try:
        df_bfs = pd.read_csv(BFS_FILE)
        KANTON_POP = dict(zip(df_bfs['Kanton'], df_bfs['Bevoelkerung']))
        print(f'BFS geladen: {len(KANTON_POP)} Kantone (aus PROD-Cache)')
    except Exception:
        KANTON_POP = KANTON_POP_FALLBACK
        print('BFS Cache Fehler — Fallback 2023 aktiv')
else:
    KANTON_POP = KANTON_POP_FALLBACK
    print(f'BFS Cache nicht gefunden ({BFS_FILE})')
    print('→ K_01 zuerst ausführen, oder: python -c "..." für Download')

print(f'Total: {sum(KANTON_POP.values()):,} Einw. | GRANULARITY={GRANULARITY}')


---
## 4. Neue Zonenzuweisung (6 Zonen)<a id='4-zonenzuweisung_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

### 4.1 Zonendefinition

Alle Variablen in diesem Block tragen das Suffix `_B` — sie koexistieren mit dem K_01-Modell
und überschreiben **keine** globalen Variablen.

| Zone | Kantone | Physikalische Rolle | Änderung vs. K_01 |
|------|---------|--------------------|--------------------|
| **Nordost** | ZH, TG, SH, SG | Lastzentrum Ost-Mittelland, grösster Nettoimporteur | K_01 «Nord» ohne AR/AI |
| **Ostschweiz** | AR, AI, GR, GL, SZ | Alpine Ostschweiz, Kleinwasserkraft + Tourismus | K_01 «Nord» (AR/AI) + «Ost» (GR/GL/SZ) ohne UR |
| **Mitte-Erzeugung** | AG, SO, BL | AKW-Gürtel, Produktionsüberschuss | K_01 «Mitte» geteilt |
| **Mitte-Transit** | BE, LU, BS, ZG, NW, OW | Transit + Urban, ausgeglichen | K_01 «Mitte» geteilt |
| **West** | VD, GE, NE, JU, FR | Romandie, Cross-Border FR | unverändert |
| **Süd** | VS, TI, **UR** | Wasserkraft + Gotthard-Korridor | UR neu hier (war «Ost») |

> ⚙ Alle Werte in diesem Block sind als potenzielle config.json-Keys markiert.
> Vollständige Dokumentation → Sektion 7.


### ⚙ Granularitäts-Hinweis: Kantone vs. Gemeinden vs. NUTS-3

Die aktuelle Zonierung basiert auf **26 Kantonen** als kleinste Einheit.
Für präzisere Analysen stehen feinere Auflösungen zur Verfügung:

| Ebene | Einheiten CH | Datenquelle | Verfügbar via |
|-------|-------------|-------------|---------------|
| **Kantone** | 26 | BFS STATPOP | PXWeb API ✅ |
| **Bezirke** | ~150 | BFS STATPOP | PXWeb API ✅ |
| **Gemeinden** | ~2'100 | BFS STATPOP + swisstopo | PXWeb API + swissBOUNDARIES3D ✅ |
| **NUTS-3** | 26 (= Kantone für CH) | Eurostat | GISCO GeoJSON ✅ |
| **Baublöcke** | ~150'000 | swisstopo | Kein API — zu granular für Verbrauchsmodell |

→ Für internationale Vergleiche: NUTS-2/3-Auflösung via Eurostat (→ Konzept-Markdown)
→ Für CH: BFS PXWeb liefert Gemeindedaten im gleichen Format — Code bereits generisch


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ⚠️  LOKALE ZONENDEFINITION — NICHT IN CONFIG.JSON
# Suffix _B: alle Variablen koexistieren mit K_01-Variablen
# ⚙ = sollte nach Projekt-Abschluss in config.json unter kuer.k01b.*
# ═══════════════════════════════════════════════════════════════════════════════

# ⚙  config.json → kuer.k01b.kanton_to_zone
KANTON_TO_ZONE_B = {
    # Nordost: Lastzentrum Zürich-Region + Bodensee
    'ZH':'Nordost','TG':'Nordost','SH':'Nordost','SG':'Nordost',
    # Ostschweiz: Alpine Kantone + AR/AI (bisher in Nord)
    'AR':'Ostschweiz','AI':'Ostschweiz',
    'GR':'Ostschweiz','GL':'Ostschweiz','SZ':'Ostschweiz',
    # Mitte-Erzeugung: AKW-Gürtel (Beznau/Gösgen/Leibstadt + Mühleberg stillgelegt)
    'AG':'Mitte-Erzeugung','SO':'Mitte-Erzeugung','BL':'Mitte-Erzeugung',
    # Mitte-Transit: Transitzone + Urban
    'BE':'Mitte-Transit','LU':'Mitte-Transit','BS':'Mitte-Transit',
    'ZG':'Mitte-Transit','NW':'Mitte-Transit','OW':'Mitte-Transit',
    # West: Romandie (unverändert)
    'VD':'West','GE':'West','NE':'West','JU':'West','FR':'West',
    # Süd: Wasserkraft + UR (Gotthard-Korridor — netztechnisch Süd-Achse)
    'VS':'Süd','TI':'Süd','UR':'Süd',
}

# ⚙  config.json → kuer.k01b.zone_colors
ZONE_COLORS_B = {
    'Nordost'        : '#1565C0',  # Dunkelblau  — Lastzentrum
    'Ostschweiz'     : '#00838F',  # Teal        — Alpenwasserkraft
    'Mitte-Erzeugung': '#7B1FA2',  # Violett     — AKW-Erzeugung
    'Mitte-Transit'  : '#388E3C',  # Dunkelgrün  — Transit/Urban
    'West'           : '#FF6F00',  # Amber       — Grenzengpass FR
    'Süd'            : '#B71C1C',  # Dunkelrot   — Überschusszone
}

# ⚙  config.json → kuer.k01b.engpass_multiplikator
ENGPASS_MULT_B = {
    'Nordost'        : 1.6,   # Höchstes Verbrauchsdefizit + Mittelland-Engpass
    'Ostschweiz'     : 1.4,   # Nord-Süd-Transitfunktion (SZ, GR)
    'Mitte-Erzeugung': 1.2,   # AKW-Überschuss → muss abgeleitet werden
    'Mitte-Transit'  : 0.9,   # Gering — kaum kritische Engpässe
    'West'           : 1.8,   # FR-CH Grenzengpass (PST-massnahmen aktiv)
    'Süd'            : 2.2,   # Göschenen-Airolo + Gotthard: kritischster CH-Engpass
}

# ⚙  config.json → kuer.k01b.zone_bottleneck (Beschreibung)
ZONE_BOTTLENECK_B = {
    'Nordost'        : 'Grösstes Verbrauchsdefizit CH; AG-ZH Mittelland-Engpass',
    'Ostschweiz'     : 'Kleinwasserkraft, Nord-Süd-Transit via GR/SZ',
    'Mitte-Erzeugung': 'AKW-Überschuss (Beznau, Gösgen, Leibstadt); Nettoexport',
    'Mitte-Transit'  : 'Transitzone Bern-Luzern; Urban BS/ZG; ausgeglichen',
    'West'           : 'Import/Export Frankreich; PST-gesteuerte Grenzflüsse',
    'Süd'            : 'Wasserkraft-Überschuss VS+TI; Flaschenhals Göschenen-Airolo+Gotthard',
}

# ⚙  config.json → kuer.k01b.spez_kw_person (identisch K_01)
SPEZ_KW_PERSON_B = 0.76   # kW Mittellast pro Person (BFE Elektrizitätsstatistik 2023)

# Zone-Anlagen verknüpfen
if gdf_plants is not None and COL.get('kanton'):
    gdf_plants['Zone_B'] = gdf_plants[COL['kanton']].map(KANTON_TO_ZONE_B)
    mapped = gdf_plants['Zone_B'].notna().sum()
    total  = len(gdf_plants)
    print(f'Zone_B-Mapping: {mapped:,}/{total:,} Anlagen zugeordnet')
    unmapped = gdf_plants[gdf_plants['Zone_B'].isna()][COL['kanton']].value_counts()
    if not unmapped.empty:
        print(f'  Nicht zugeordnet: {unmapped.to_dict()}')
else:
    print('⚠️  gdf_plants nicht verfügbar — Zone_B-Zuweisung übersprungen')

print('\nZonen-Überblick:')
for z, col in ZONE_COLORS_B.items():
    kts = [k for k,v in KANTON_TO_ZONE_B.items() if v==z]
    pop = sum(KANTON_POP.get(k,0) for k in kts)
    print(f'  {z:<16}: {", ".join(kts):<30} → {pop/1e6:.2f} Mio. Einw.')


---
## 5. Zonenbilanzen & BVI<a id='5-bilanzen-bvi_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

### 5.1 df_zones_b aufbauen

Identische Methodik wie K_01 (CF-gewichtete Produktion, Bevölkerung × Spezifischer Verbrauch).
Suffix `_b` auf allen Variablen — kein Konflikt mit K_01.


In [ ]:
# ── Kapazitätsfaktoren (identisch K_01 — SSOT-Kandidat) ─────────────────────
# ⚙  config.json → kuer.k01b.kapazitaetsfaktoren (oder zentral unter simulation.cf_faktoren)
CF_B = {
    'Wasserkraft':0.38,'Solar':0.12,'Wind':0.22,'Kernkraft':0.80,
    'Biomasse':0.55,'Erdgas':0.30,'Abfall':0.65,'Geothermie':0.80,
    'Erdoel':0.15,'Kohle':0.50,'Andere':0.40,
}

ZONEN_B = list(ZONE_COLORS_B.keys())

# ── Mittlere Produktion pro Zone_B ───────────────────────────────────────────
if gdf_plants is not None and 'Zone_B' in gdf_plants.columns:
    zone_et_b = (gdf_plants[gdf_plants['Zone_B'].notna()]
                 .groupby(['Zone_B','ET_group'])['kw'].sum().reset_index())
    zone_et_b['CF']       = zone_et_b['ET_group'].map(CF_B).fillna(0.40)
    zone_et_b['mittl_MW'] = zone_et_b['kw'] / 1000 * zone_et_b['CF']
    zone_prod_b = zone_et_b.groupby('Zone_B')['mittl_MW'].sum()
    zone_inst_b = (gdf_plants[gdf_plants['Zone_B'].notna()]
                   .groupby('Zone_B')['kw'].sum() / 1000)
else:
    zone_prod_b = pd.Series({z: 0 for z in ZONEN_B})
    zone_inst_b = zone_prod_b.copy()
    print('⚠️  gdf_plants fehlt — Nullwerte für Produktion')

# ── Bevölkerung & Verbrauch pro Zone_B ───────────────────────────────────────
zone_pop_b = {}
for k, z in KANTON_TO_ZONE_B.items():
    zone_pop_b[z] = zone_pop_b.get(z, 0) + KANTON_POP.get(k, 0)

df_zones_b = pd.DataFrame({
    'Zone'         : ZONEN_B,
    'Inst_MW'      : [float(zone_inst_b.get(z, 0)) for z in ZONEN_B],
    'Produktion_MW': [float(zone_prod_b.get(z, 0)) for z in ZONEN_B],
    'Bevoelkerung' : [zone_pop_b.get(z, 0)          for z in ZONEN_B],
    'Engpass'      : [ZONE_BOTTLENECK_B[z]           for z in ZONEN_B],
})
df_zones_b['Verbrauch_MW'] = df_zones_b['Bevoelkerung'] * SPEZ_KW_PERSON_B / 1000
df_zones_b['Imbalance_MW'] = df_zones_b['Produktion_MW'] - df_zones_b['Verbrauch_MW']

# ── BVI (Battery Value Index) ─────────────────────────────────────────────────
df_zones_b['BVI_raw']  = (df_zones_b['Imbalance_MW'].abs() *
                           df_zones_b['Zone'].map(ENGPASS_MULT_B))
df_zones_b['BVI_norm'] = df_zones_b['BVI_raw'] / df_zones_b['BVI_raw'].sum() * 10

print('\ndf_zones_b (6-Zonen-Modell):')
print(df_zones_b[['Zone','Verbrauch_MW','Produktion_MW','Imbalance_MW','BVI_norm']].to_string(index=False))


**Verifikation:** Shape, Nullwerte und Wertebereich.

In [ ]:
# ── Verifikation ─────────────────────────────────────────────────────────────
print(f'Shape    : {df_zones_b.shape}')
print(f'Nullwerte: {df_zones_b.isnull().sum().sum()}')
print(f'BVI-Summe: {df_zones_b["BVI_norm"].sum():.2f} (sollte 10.0)')
print(f'Imbalance-Range: {df_zones_b["Imbalance_MW"].min():.0f} – {df_zones_b["Imbalance_MW"].max():.0f} MW')
df_zones_b[['Zone','Verbrauch_MW','Produktion_MW','Imbalance_MW','BVI_norm','Engpass']]


### 5.2 Charts: Zonenimbalance & BVI


In [ ]:
print("▶ Starte Erstellung: Zonenimbalance K_01b (6 Zonen)")
# 🔬 MODELLIERT — Zonen-Imbalance aus BFE-Kapazitäten × CF × saisonalem Profil vs. BFS-Bevölkerungsverbrauch
# ── Chart K01b-1: Zonenimbalance Balkendiagramm (6 Zonen) ───────────────────
bar_colors_b = [ZONE_COLORS_B[z] for z in df_zones_b.sort_values('Verbrauch_MW')['Zone']]
df_s = df_zones_b.sort_values('Verbrauch_MW', ascending=True)
bar_colors_b = [ZONE_COLORS_B[z] for z in df_s['Zone']]

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
ax.tick_params(colors=C_TICK)

ax.barh(df_s['Zone'], df_s['Verbrauch_MW'], color=bar_colors_b, alpha=0.85,
        label='Verbrauch (Mittellast)')
ax.barh(df_s['Zone'], df_s['Produktion_MW'], color='none', alpha=0.9,
        edgecolor='white', linewidth=1.5, linestyle='--',
        label='Produktion (CF-gewichtet)')

for _, row in df_s.iterrows():
    imb = row['Imbalance_MW']
    col = C_LOAD if imb > 0 else C_PRICE
    xpos = max(row['Verbrauch_MW'], row['Produktion_MW']) + 30
    ax.text(xpos, row['Zone'], f'{imb:+.0f} MW',
            va='center', color=col, fontsize=FS_ACHSE, fontweight='bold')

ax.set_xlabel('MW', color=C_ACHSE)
ax.set_title('Mittellast-Verbrauch vs. CF-Produktion — 6-Zonen-Modell\n'
             '+/– = Netto-Imbalance [MW]',
             color='white', fontsize=FS_TITEL, fontweight='bold')
ax.legend(fontsize=FS_TICK, framealpha=ALPHA_LEGENDE,
          facecolor=C_LEGENDE_BG, labelcolor='white')
ax.grid(True, axis='x', alpha=ALPHA_FLAECHE)

plt.tight_layout()
p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_zonenimbalance.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
print(f'Gespeichert: {p}')


In [ ]:
# ── Ausgabe ──────────────────────────────────────────────────────
from IPython.display import Image, display
display(Image(filename=os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_zonenimbalance.png')))


**Zonenimbalance K_01b (6 Zonen)**

Imbalance-Balken: Zonen mit positivem Wert (grüner Ring) sind Netto-Exporteure, negative Defizit-Zonen.
Die **Mitte-Erzeugung**-Zone (AG/SO/BL) hebt sich als klarer AKW-Überschuss-Knoten ab.
**Nordost** (ZH-Raum) zeigt das grösste Verbrauchsdefizit — hoher BVI-Wert.

In [ ]:
print("▶ Starte Erstellung: BVI K_01b (6 Zonen)")
# 🔬 MODELLIERT — BVI aus Imbalance × Engpass-Multiplikator (⚙ kuer.k01b.engpass_multiplikator)
# ── Chart K01b-2: BVI Balken (6 Zonen) ───────────────────────────────────────
df_bvi = df_zones_b.sort_values('BVI_norm', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
ax.tick_params(colors=C_TICK)

bvi_colors = [ZONE_COLORS_B[z] for z in df_bvi['Zone']]
bars = ax.barh(df_bvi['Zone'], df_bvi['BVI_norm'], color=bvi_colors, alpha=0.85)

for bar, (_, row) in zip(bars, df_bvi.iterrows()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{row["BVI_norm"]:.2f} / 10  |  Mult. {ENGPASS_MULT_B[row["Zone"]]}×',
            va='center', color='white', fontsize=FS_LEGENDE)

ax.set_xlabel('BVI (normiert auf 10)', color=C_ACHSE)
ax.set_title('Battery Value Index — 6-Zonen-Modell (K_01b)\n'
             'Imbalance × Engpass-Multiplikator, normiert',
             color='white', fontsize=FS_TITEL, fontweight='bold')
ax.set_xlim(0, df_bvi['BVI_norm'].max() * 1.45)
ax.grid(True, axis='x', alpha=ALPHA_FLAECHE)

plt.tight_layout()
p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_bvi.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
print(f'Gespeichert: {p}')


In [ ]:
# ── Ausgabe ──────────────────────────────────────────────────────
from IPython.display import Image, display
display(Image(filename=os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_bvi.png')))


**BVI K_01b (6 Zonen)**

Battery Value Index normiert auf 10. **Süd** und **West** haben die höchsten BVI-Werte wegen
starker physikalischer Engpässe (Göschenen-Airolo, FR-CH-Grenze).
Mitte-Transit zeigt den tiefsten BVI — wenig Produktionsengpass, ausgeglichene Bilanz.

---
## 6. Vergleich: 5-Zonen (K_01) vs. 6-Zonen (K_01b)<a id='6-vergleich_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

### 6.1 Hauptunterschiede


| Dimension | K_01 (5 Zonen) | K_01b (6 Zonen) | Verbesserung |
|-----------|---------------|----------------|-------------|
| «Nord» | ZH, TG, SH, AR, AI, SG | → **Nordost** (ZH, TG, SH, SG) | Lastzentrum sauber isoliert |
| «Ost» | GR, GL, UR, SZ | → **Ostschweiz** (AR, AI, GR, GL, SZ) | AR/AI korrekt zugeordnet |
| «Mitte» | 9 Kantone (heterogen) | → **M-Erzeugung** (AG, SO, BL) + **M-Transit** (BE, LU, BS, ZG, NW, OW) | AKW-Zone isoliert |
| UR | In «Ost» | In **«Süd»** | Gotthard-Korridor korrekt |
| «West» | unverändert | unverändert | — |
| «Süd» | VS, TI | VS, TI, **UR** | Süd-Achse vollständig |
| BVI-Granularität | AG/BE ununterscheidbar | M-Erzeugung vs. M-Transit getrennt | Investitionsentscheid verbessert |

### 6.2 BVI-Konsequenzen


In [ ]:
print("▶ Starte Erstellung: BVI-Vergleich 5 vs. 6 Zonen")
# 🔬 MODELLIERT — BVI-Vergleich 5 Zonen (K_01) vs. 6 Zonen (K_01b)
# ── Chart K01b-3: BVI Vergleich 5- vs 6-Zonen ────────────────────────────────
# Manuell rekonstruierter K_01-BVI aus bekannten Werten (Transfer-unabhängig)
k01_bvi_approx = {
    'Nord' : 1.8, 'Mitte': 2.4, 'West': 2.1, 'Süd': 3.3, 'Ost': 0.4
}  # Approximation aus letztem K_01-Run; exakt nach K_01-Ausführung

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(BG_DARK)

k01_colors = ['#1565C0','#388E3C','#FF6F00','#B71C1C','#00838F']
k01_zones  = list(k01_bvi_approx.keys())
k01_vals   = list(k01_bvi_approx.values())

for ax in axes:
    ax.set_facecolor(BG_PANEL)
    for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)
    ax.tick_params(colors=C_TICK)

# K_01 (5 Zonen)
bars1 = axes[0].bar(k01_zones, k01_vals, color=k01_colors, alpha=0.85)
for bar, val in zip(bars1, k01_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}', ha='center', color='white', fontsize=9, fontweight='bold')
axes[0].set_title('K_01: BVI — 5 Zonen (Referenz)', color='white', fontsize=12)
axes[0].set_ylabel('BVI (normiert auf 10)', color=C_ACHSE)
axes[0].axhline(2.0, color=C_PRICE, lw=1, linestyle=':', alpha=0.7, label='Ø 2.0')
axes[0].legend(fontsize=FS_TICK, framealpha=0.3, facecolor=C_LEGENDE_BG, labelcolor='white')
axes[0].grid(True, axis='y', alpha=ALPHA_FLAECHE)

# K_01b (6 Zonen)
df_b_sorted = df_zones_b.sort_values('BVI_norm', ascending=False)
b_colors    = [ZONE_COLORS_B[z] for z in df_b_sorted['Zone']]
bars2 = axes[1].bar(df_b_sorted['Zone'], df_b_sorted['BVI_norm'],
                    color=b_colors, alpha=0.85)
for bar, (_, row) in zip(bars2, df_b_sorted.iterrows()):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{row["BVI_norm"]:.2f}', ha='center',
                 color='white', fontsize=9, fontweight='bold')
axes[1].set_title('K_01b: BVI — 6 Zonen (Erweitert)', color='white', fontsize=12)
axes[1].set_ylabel('BVI (normiert auf 10)', color=C_ACHSE)
axes[1].axhline(10/6, color=C_PRICE, lw=1, linestyle=':', alpha=0.7, label='Ø 1.67')
axes[1].legend(fontsize=FS_TICK, framealpha=0.3, facecolor=C_LEGENDE_BG, labelcolor='white')
axes[1].grid(True, axis='y', alpha=ALPHA_FLAECHE)

for ax in axes: ax.tick_params(axis='x', rotation=20)

plt.suptitle('BVI-Vergleich: 5-Zonen-Modell (K_01) vs. 6-Zonen-Modell (K_01b)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
p = os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_bvi_vergleich.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
print(f'Gespeichert: {p}')


In [ ]:
# ── Ausgabe ──────────────────────────────────────────────────────
from IPython.display import Image, display
display(Image(filename=os.path.join(EXP_CHARTS_DIR, 'EXP_kuer_k01b_bvi_vergleich.png')))


**BVI-Vergleich 5 vs. 6 Zonen**

Das 6-Zonen-Modell macht die funktionale Heterogenität der ehemaligen «Mitte»-Zone sichtbar:
Mitte-Erzeugung (AKW-Gürtel) und Mitte-Transit (Transitzone) haben sehr unterschiedliche BVI-Werte.
Für Investitionsentscheide auf Gemeindeebene empfiehlt sich NUTS-3-Auflösung (→ Konzept).

**Interpretation Vergleich:**

Das 6-Zonen-Modell macht zwei Effekte sichtbar, die im 5-Zonen-Modell verborgen bleiben:

1. **Mitte-Erzeugung vs. Mitte-Transit:** AG/SO/BL als AKW-Erzeugungszone hat einen hohen
   Produktionsüberschuss und damit hohen BVI — ein Batteriespeicher hier puffert AKW-Baseload
   statt Transit zu decken. Mitte-Transit hat dagegen ausgeglichene Bilanz und tiefen BVI.

2. **Nordost isoliert:** Mit AR/AI und GR/GL/SZ ausgegliedert verliert «Nordost» etwas Bevölkerung
   aber behält das gesamte Verbrauchsdefizit der Zürich-Region — der BVI steigt, weil der
   Engpassmultiplikator nun auf eine konzentriertere Lastsituation trifft.

3. **Süd gewinnt UR:** Der Gotthard-Korridor (UR) ist physikalisch Teil des Süd-Engpasses.
   Die Imbalance von Süd steigt, der BVI ebenfalls — konsistenter mit Swissgrid-Netzanalysen.


---
## 7. ⚙ Config-Dokumentation: Was in config.json gehört<a id='7-config-doku_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

> Diese Sektion dokumentiert alle lokalen Werte dieses Notebooks die bei einer
> Config-Integration unter `config.json → kuer → k01b` eingetragen werden sollten.
> Kein Code hier — reine Dokumentation für den späteren Integrations-Schritt.

### 7.1 Vorgeschlagene JSON-Struktur

```json
"kuer": {
  "k01b": {
    "_beschreibung": "6-Zonen-Erweiterung des K_01 Zonenmodells — explorativ",
    "_status": "provisorisch, nicht in K_00 referenziert",

    "kanton_to_zone": {
      "ZH": "Nordost",  "TG": "Nordost",  "SH": "Nordost",  "SG": "Nordost",
      "AR": "Ostschweiz","AI": "Ostschweiz","GR": "Ostschweiz",
      "GL": "Ostschweiz","SZ": "Ostschweiz",
      "AG": "Mitte-Erzeugung","SO": "Mitte-Erzeugung","BL": "Mitte-Erzeugung",
      "BE": "Mitte-Transit","LU": "Mitte-Transit","BS": "Mitte-Transit",
      "ZG": "Mitte-Transit","NW": "Mitte-Transit","OW": "Mitte-Transit",
      "VD": "West","GE": "West","NE": "West","JU": "West","FR": "West",
      "VS": "Süd","TI": "Süd","UR": "Süd"
    },

    "zone_colors": {
      "Nordost"        : "#1565C0",
      "Ostschweiz"     : "#00838F",
      "Mitte-Erzeugung": "#7B1FA2",
      "Mitte-Transit"  : "#388E3C",
      "West"           : "#FF6F00",
      "Süd"            : "#B71C1C"
    },

    "engpass_multiplikator": {
      "Nordost"        : 1.6,
      "Ostschweiz"     : 1.4,
      "Mitte-Erzeugung": 1.2,
      "Mitte-Transit"  : 0.9,
      "West"           : 1.8,
      "Süd"            : 2.2
    },

    "spez_kw_person": 0.76
  }
}
```

### 7.2 Integrations-Checkliste (für nach Projekt-Abgabe)

| Schritt | Was | Betroffene Notebooks |
|---------|-----|---------------------|
| 1 | `kuer.k01b`-Block in config.json eintragen | — |
| 2 | K_01b: alle `_B`-Konstanten durch `CFG['kuer']['k01b'][...]` ersetzen | K_01b |
| 3 | K_01 weiterhin mit `kuer.k01`-Block (oder Basis-Config) | K_01 |
| 4 | K_00: Sektion 5 um K_01b-Charts ergänzen (optional) | K_00 |
| 5 | O_02 Glossar: «Nordost», «Ostschweiz», «Mitte-Erzeugung», «Mitte-Transit» als Projektbegriffe | O_02 |
| 6 | O_03 Konventionen: Chart-Prefix `kuer_k01b_` dokumentieren | O_03 |

### 7.3 Wichtiger Hinweis: Rückwärtskompatibilität

Die Umbenennung von `'Nord'` → `'Nordost'` etc. würde alle bestehenden K_01-Charts
ungültig machen und Kaskaden-Updates in K_00, K_99, O_99 auslösen.
**Empfehlung:** K_01 und K_01b dauerhaft parallel betreiben (verschiedene Zone-Namespaces),
statt K_01 zu ersetzen.


---
## Abschluss<a id='abschluss_K_01b'></a>

[↑ Inhaltsverzeichnis](#toc_K_01b)

Abschlusskontrolle: alle erwarteten Charts vorhanden?


In [ ]:
# ── Abschlusskontrolle K_01b ─────────────────────────────────────────────────
print('K_01b – Abschlusskontrolle')
print('=' * 60)
expected = [
    'EXP_kuer_k01b_zonenimbalance.png',
    'EXP_kuer_k01b_bvi.png',
    'EXP_kuer_k01b_bvi_vergleich.png',
]
all_ok = True
for fname in expected:
    p = os.path.join(EXP_CHARTS_DIR, fname)
    ok = os.path.exists(p)
    print(f'  {"✅" if ok else "❌"} {fname}')
    if not ok: all_ok = False
print()
print('Lokale Zonen (K_01b):')
for z in ZONEN_B:
    n_anl = (len(gdf_plants[gdf_plants['Zone_B']==z])
             if gdf_plants is not None and 'Zone_B' in gdf_plants.columns else '?')
    pop   = zone_pop_b.get(z, 0) / 1e6
    bvi   = df_zones_b[df_zones_b['Zone']==z]['BVI_norm'].values[0]
    print(f'  {z:<16}: {n_anl:>6} Anlagen | {pop:.2f} Mio. | BVI {bvi:.2f}')
print()
print('→ Dieses Notebook ist NICHT in K_00 Business Strategy referenziert.')
print('→ Config-Integration: nach Projekt-Abgabe (Sektion 7).')
if all_ok:
    print('✅ Alle Charts vorhanden.')
else:
    print('❌ Fehlende Charts — Notebook erneut ausführen.')


| [← K_01 Räumliche Analyse](K_01_Raeumliche_Analyse.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_02 Cross-Border →](K_02_Cross_Border.ipynb) |
|:---|:---:|---:|